# Education Analytics Dashboard
Interactive KPI summary for Student Outcomes, Course Performance, and Institutional Health.

In [ ]:
df_scorecards  = spark.read.format('delta').table('gold_student_scorecards').limit(100000).toPandas()
df_course      = spark.read.format('delta').table('gold_course_performance').limit(100000).toPandas()
df_faculty     = spark.read.format('delta').table('gold_faculty_workload').limit(100000).toPandas()
df_cohort      = spark.read.format('delta').table('gold_cohort_outcomes').limit(100000).toPandas()

In [ ]:
total_students   = len(df_scorecards)
active_students  = int(df_scorecards['is_active'].sum())
withdrawn        = int(df_scorecards['is_withdrawn'].sum())
avg_score        = round(df_scorecards['avg_score'].mean(), 1)
overall_pass_rate = round(df_scorecards['pass_rate'].mean(), 1)
at_risk_count    = int((df_scorecards['retention_risk'] == 'High').sum())

print(f"Total Students    : {total_students:,}")
print(f"Active Students   : {active_students:,}")
print(f"Withdrawn         : {withdrawn:,}")
print(f"Avg Score         : {avg_score}")
print(f"Overall Pass Rate : {overall_pass_rate}%")
print(f"High Risk Students: {at_risk_count:,}")

In [ ]:
# Bottom 10 courses by pass rate
low_courses = (
    df_course[['course_id', 'department', 'level', 'assessment_type',
               'total_submissions', 'avg_score', 'pass_rate', 'resit_rate']]
    .sort_values('pass_rate')
    .head(10)
)

# Department summary
dept_summary = (
    df_cohort.groupby('department').agg(
        total_students=('total_students', 'sum'),
        avg_score=('cohort_avg_score', 'mean'),
        avg_pass_rate=('cohort_pass_rate', 'mean'),
        avg_retention=('retention_rate', 'mean'),
    ).reset_index()
    .sort_values('avg_score', ascending=False)
)
dept_summary['avg_score']     = dept_summary['avg_score'].round(1)
dept_summary['avg_pass_rate'] = dept_summary['avg_pass_rate'].round(1)
dept_summary['avg_retention'] = dept_summary['avg_retention'].round(1)

html = f"""
<!DOCTYPE html><html><head>
<meta charset="utf-8">
<title>Education Analytics Dashboard</title>
<style>
  body {{ font-family: Segoe UI, Arial, sans-serif; background:#f4f6f9; margin:0; padding:20px; }}
  h1   {{ color:#0078d4; border-bottom:3px solid #0078d4; padding-bottom:8px; }}
  h2   {{ color:#323130; margin-top:30px; }}
  .kpis {{ display:flex; gap:16px; flex-wrap:wrap; margin:20px 0; }}
  .kpi  {{ background:#fff; border-radius:8px; padding:20px 28px; box-shadow:0 2px 6px rgba(0,0,0,.08);
           min-width:150px; text-align:center; }}
  .kpi .val {{ font-size:2em; font-weight:700; color:#0078d4; }}
  .kpi .lbl {{ font-size:.85em; color:#605e5c; margin-top:4px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff; border-radius:8px;
           box-shadow:0 2px 6px rgba(0,0,0,.08); overflow:hidden; margin-bottom:30px; }}
  th    {{ background:#0078d4; color:#fff; padding:10px 14px; text-align:left; font-size:.9em; }}
  td    {{ padding:9px 14px; border-bottom:1px solid #edebe9; font-size:.88em; }}
  tr:hover td {{ background:#f3f9ff; }}
  .badge {{ display:inline-block; padding:2px 10px; border-radius:12px; font-size:.8em; font-weight:600; }}
  .Distinction {{ background:#dff6dd; color:#107c10; }}
  .Merit       {{ background:#cceaff; color:#0078d4; }}
  .Pass        {{ background:#e8f5e9; color:#388e3c; }}
  .Near.Pass   {{ background:#fff4ce; color:#b7610e; }}
  .Fail        {{ background:#fde7e9; color:#a80000; }}
  .High        {{ background:#fde7e9; color:#a80000; }}
  .Low         {{ background:#dff6dd; color:#107c10; }}
</style></head><body>
<h1>🎓 Education Analytics Dashboard</h1>

<div class="kpis">
  <div class="kpi"><div class="val">{total_students:,}</div><div class="lbl">Total Students</div></div>
  <div class="kpi"><div class="val">{active_students:,}</div><div class="lbl">Active Students</div></div>
  <div class="kpi"><div class="val">{overall_pass_rate}%</div><div class="lbl">Overall Pass Rate</div></div>
  <div class="kpi"><div class="val">{avg_score}</div><div class="lbl">Avg Score</div></div>
  <div class="kpi"><div class="val">{withdrawn:,}</div><div class="lbl">Withdrawals</div></div>
  <div class="kpi"><div class="val">{at_risk_count:,}</div><div class="lbl">High-Risk Students</div></div>
</div>

<h2>Lowest-Performing Courses (Bottom 10 by Pass Rate)</h2>
<table>
  <tr><th>Course ID</th><th>Department</th><th>Level</th><th>Assessment Type</th><th>Submissions</th><th>Avg Score</th><th>Pass Rate</th><th>Resit Rate</th></tr>
"""
for _, r in low_courses.iterrows():
    html += f"""
  <tr>
    <td>{r['course_id']}</td>
    <td>{r['department']}</td>
    <td>{r['level']}</td>
    <td>{r['assessment_type']}</td>
    <td>{int(r['total_submissions']):,}</td>
    <td>{r['avg_score']:.1f}</td>
    <td>{r['pass_rate']:.1f}%</td>
    <td>{r['resit_rate']:.1f}%</td>
  </tr>"""

html += """
</table>

<h2>Department Performance Summary</h2>
<table>
  <tr><th>Department</th><th>Total Students</th><th>Avg Score</th><th>Pass Rate</th><th>Retention Rate</th></tr>
"""
for _, r in dept_summary.iterrows():
    html += f"""
  <tr>
    <td>{r['department']}</td>
    <td>{int(r['total_students']):,}</td>
    <td>{r['avg_score']}</td>
    <td>{r['avg_pass_rate']}%</td>
    <td>{r['avg_retention']}%</td>
  </tr>"""

html += """
</table>
</body></html>"""

displayHTML(html)

In [ ]:
mssparkutils.fs.put('Files/dashboard.html', html, overwrite=True)
print('Dashboard saved to Files/dashboard.html')